# 00-Cleanup — Remove Old Test Tenants from Neptune

This notebook helps you clean stale test data from previous notebook runs.

**What it does:**
1. Connects to Neptune and lists all node labels (which encode tenant IDs)
2. Identifies tenants you want to remove vs. keep
3. Deletes nodes belonging to old tenants

**Tenant label format:** Labels follow the graphrag-toolkit convention:
- Default tenant: `` `Label` ``
- Custom tenant: `` `Label{tenant_id}__` `` (e.g., `Persondemo__`)

**⚠️ Warning:** This notebook permanently deletes graph data. Review the tenant list before running the delete cell.

## Configuration

In [ ]:
import os
import re

# Replace with your Neptune endpoint:
GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)

# Tenants to KEEP (will not be deleted):
KEEP_TENANTS = ['hybrid_demo', 'mandela']

print(f'Endpoint: {GRAPH_STORE}')
print(f'Protected tenants: {KEEP_TENANTS}')

## Connect to Neptune and List Labels

In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
gs = graph_store.__enter__()

result = gs.execute_query(
    'MATCH (n) RETURN DISTINCT labels(n) as lbl, count(n) as cnt ORDER BY cnt DESC LIMIT 50'
)

print(f'Found {len(result)} distinct label sets in Neptune:\n')
for row in result:
    print(f'  {row["lbl"][0]:40s} {row["cnt"]:>6} nodes')

## Identify Tenants

Extract tenant IDs from labels. The tenant format is `Label{tenant_id}__` — we parse the suffix.

In [ ]:
# Pattern: label ends with {tenant_id}__ where tenant_id is 1-25 lowercase alphanum/periods
TENANT_PATTERN = re.compile(r'^(.+?)([a-z0-9.]{1,25})__$')

all_tenants = set()
for row in result:
    label = row['lbl'][0]
    match = TENANT_PATTERN.match(label)
    if match:
        all_tenants.add(match.group(2))

tenants_to_remove = all_tenants - set(KEEP_TENANTS)
tenants_to_keep = all_tenants & set(KEEP_TENANTS)

print(f'All tenants found: {sorted(all_tenants)}')
print(f'Will KEEP:   {sorted(tenants_to_keep)}')
print(f'Will REMOVE: {sorted(tenants_to_remove)}')

if not tenants_to_remove:
    print('\n✅ Nothing to clean up.')

## Delete Old Tenants

⚠️ **This permanently deletes data.** Review the list above before running this cell.

In [ ]:
# Safety check — set to True to actually delete
CONFIRM_DELETE = False

if not CONFIRM_DELETE:
    print('⚠️  CONFIRM_DELETE is False. Set to True and re-run to delete.')
    print(f'   Would delete tenants: {sorted(tenants_to_remove)}')
else:
    for tenant in sorted(tenants_to_remove):
        # Find all nodes whose label ends with {tenant}__
        suffix = f'{tenant}__'
        count_q = f"MATCH (n) WHERE any(l IN labels(n) WHERE l ENDS WITH '{suffix}') RETURN count(n) as cnt"
        count_result = gs.execute_query(count_q)
        cnt = count_result[0]['cnt'] if count_result else 0

        if cnt > 0:
            delete_q = f"MATCH (n) WHERE any(l IN labels(n) WHERE l ENDS WITH '{suffix}') DETACH DELETE n"
            gs.execute_query(delete_q)
            print(f'  🗑️  Deleted {cnt} nodes from tenant: {tenant}')
        else:
            print(f'  ⏭️  {tenant}: already empty')

    print('\n✅ Cleanup complete.')

## Verify — Show Remaining Data

In [ ]:
result = gs.execute_query(
    'MATCH (n) RETURN DISTINCT labels(n) as lbl, count(n) as cnt ORDER BY cnt DESC LIMIT 30'
)

print(f'Remaining labels ({len(result)} distinct):\n')
for row in result:
    print(f'  {row["lbl"][0]:40s} {row["cnt"]:>6} nodes')

# Clean up context manager
graph_store.__exit__(None, None, None)